In [63]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import math
from scipy.stats import norm
import matplotlib.pyplot as plt
import sys

In [65]:
#Entry
tperiod = 8 #time period is 4 hours long

In [67]:
# Location data
df_V = pd.read_excel('fullDist Set V Data Location.xlsx')
df_V.head

<bound method NDFrame.head of     Index      County       Type  Cluster  \
0       0         NaN  Warehouse      NaN   
1       1         NaN  Warehouse      4.0   
2       2         NaN   Supplier      NaN   
3       3         NaN   Supplier      NaN   
4       4         NaN   Supplier      NaN   
5       5         NaN   Supplier      NaN   
6       6  Georgetown    Shelter      9.0   
7       7  Georgetown    Shelter      1.0   
8       8       Horry    Shelter     18.0   
9       9       Horry    Shelter      2.0   
10     10       Horry    Shelter     16.0   
11     11       Horry    Shelter      8.0   
12     12    Berkeley    Shelter     12.0   
13     13    Berkeley    Shelter     17.0   
14     14    Berkeley    Shelter     14.0   
15     15    Berkeley    Shelter     11.0   
16     16    Berkeley    Shelter     19.0   
17     17  Charleston    Shelter      0.0   
18     18  Charleston    Shelter     15.0   
19     19  Dorchester    Shelter      7.0   
20     20  Dorchester    

In [69]:
# County data
df_C = pd.read_excel('fullDist Set C Data.xlsx')
df_C.head

<bound method NDFrame.head of        County Category of Hurricane  Shelter Total Demand  PoD Total Demand  \
0    Berkeley                 Cat 2                  6400             11596   
1  Charleston                 Cat 2                  5103              9246   
2    Colleton                 Cat 1                  1650              2500   
3  Dorchester                 Cat 1                  4460              8080   
4  Georgetown                 Cat 3                  2970              5368   
5       Horry                 Cat 3                 20234             36574   
6      Jasper                   NaN                     0                 0   

   % Evac to PubShelter  Unnamed: 5  Unnamed: 6  Unnamed: 7 All cots:  \
0                 0.190         NaN         NaN         NaN    Water    
1                 0.190         NaN         NaN         NaN     Meals   
2                 0.109         NaN         NaN         NaN       Cot   
3                 0.190         NaN         N

In [71]:
# Resource Data
df_R = pd.read_excel('fullDist R.xlsx')
df_R.head

<bound method NDFrame.head of              R  Demand Units on Truck  Pallet Units on Truck  \
0  Shelter Kit                    768                     44   
1      PoD Kit                   1728                     34   
2        Water                   2912                     26   
3        Meals                   4608                     48   
4          Cot                   3120                     52   
5      Blanket                   2496                     52   

   pallet per S-Kit  pallet per P-Kit  kit pallet meeting demand  
0          0.000000          0.000000                   0.057292  
1          0.000000          0.000000                   0.019676  
2          0.008929          0.008929                   0.000000  
3          0.010417          0.010417                   0.000000  
4          0.020833          0.000000                   0.000000  
5          0.016667          0.000000                   0.000000  >

In [73]:
#Time and distance matrices 
timeMatrix = np.loadtxt("durationFULL.txt"); # load the time duration matrix (seconds)
distanceMatrix = np.loadtxt("distanceFULL.txt");

In [75]:
df_inputs = pd.read_excel('fullDistInputs.xlsx')
df_inputs.head

<bound method NDFrame.head of    Hours in 6 days  S Curve k Value  People on Truck  Hourly Salary  \
0            144.0              0.2              4.0           15.0   
1              NaN              NaN              NaN            NaN   
2              NaN              NaN              NaN            NaN   
3              NaN              NaN              NaN            NaN   
4              NaN              NaN              NaN            NaN   
5              NaN              NaN              NaN            NaN   

   Fuel Cost Per Mile  Open Cost V0  Open Cost V1  Open Cost V2  Open Cost V3  \
0                 0.1           0.0        1000.0        1250.0         500.0   
1                 NaN           NaN           NaN           NaN           NaN   
2                 NaN           NaN           NaN           NaN           NaN   
3                 NaN           NaN           NaN           NaN           NaN   
4                 NaN           NaN           NaN           NaN    

In [77]:
# Load the K_ijt data from the Excel file
df_K = pd.read_excel('K_ijt_data_fixed.xlsx')
df_K.head()

,From_Node,To_Node,Time_Period,Capacity
0,0,0,0,98
1,0,0,1,58
2,0,0,2,101
3,0,0,3,186
4,0,0,4,132


In [79]:
# Estimated demand
file_path = 'demand_data_.xlsx'
sheet_names = ['Demand_1', 'Demand_2']

d_its = {}

for s, sheet in enumerate(sheet_names):  # Scenario index starts from 0
    df = pd.read_excel(file_path, sheet_name=sheet, header=0, index_col=0)
    
    # Fill NaN with 0 if needed
    df = df.fillna(0)
    
    demand_array = df.values

    for i in range(demand_array.shape[0]):
        actual_location = i  # Start location index from 1 instead of 0
        for t in range(demand_array.shape[1]):
            d_its[(actual_location, t, s)] = demand_array[i, t]

# Print a sample to verify the correct location
print("Sample demand data for location 6, time 1, scenario 1:", d_its[(42, 17, 0)])
print("Sample demand data for location 6, time 1, scenario 2:", d_its[(42, 17, 1)])


Sample demand data for location 6, time 1, scenario 1: 0.0
Sample demand data for location 6, time 1, scenario 2: 0.0


In [61]:
# Defining sets

# County Set
C = len(df_C)
print('All counties:', C)

# Location Sets
V = len(df_V)
print('All locations:', V)

# Subsets of Location Set (V0, V1, V2, V3)
V0 = []  # Warehouses and Suppliers
V1 = []  # Shelters
V2 = []  # RSAs (Staging Areas)
V3 = []  # PoDs (Points of Distribution)


for i in range(V):
    if df_V['Type'][i] == 'Warehouse' or df_V['Type'][i] == 'Supplier':
        V0.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'Shelter':
        V1.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'RSA':
        V2.append(df_V['Index'][i])
    elif df_V['Type'][i] == 'PoD':
        V3.append(df_V['Index'][i])

# Print lengths of subsets
print('Warehouse/Supplier locations:', len(V0), V0)
print('Shelter locations:', len(V1), V1)
print('RSA locations:', len(V2), V2)
print('PoD locations:', len(V3), V3)

# Commodity Set
R = len(df_R)
print('All resources:', R)

# Time Period Set
tperiod = 8
T = int(df_inputs['Hours in 6 days'][0] / tperiod)
print('Time periods:', T)

#Time Period Subsets
T_Bef = list(range(int(T / 2)))
T_Aft = list(range(int(T / 2), T))
print('Time Before Landfall:', T_Bef)
print('Time After Landfall:', T_Aft)

#Time period set
#T=int(df_inputs['Hours in 6 days'][0]/tperiod) #6 days for specified time period length
#print('Time periods:', T)

#T_Bef=[]
#for t in range(int(T/2)):
 #   T_Bef.append(t)
#print('Time Before:', T_Bef)

#T_Aft=[]
#for t in range(int(T/2), T):
 #   T_Aft.append(t)
#print('Time After:', T_Aft)

# Scenario Set and Scenario Probabilities
#Scenario = ['S1', 'S2']
#P = [0.5, 0.5]

# Print Scenario Set and Probabilities to verify
#print('Scenarios:', S)
#print('Scenario Probabilities:', P_s)

# Scenario Set and Scenario Probabilities
#sce = [0, 1]  # Scenario indices for Scenario 1 and Scenario 2
#P = [0.5, 0.5]  # Probabilities for each scenario

# Define scenario indices for 50 scenarios
sce = list(range(50))  # This will create a list [0, 1, 2, ..., 49]

# Define equal probabilities for each of the 50 scenarios
P = [1/50] * 50  # This will create a list with 50 elements, each with a value of 1/50

# Print to verify
print("Scenario indices:", sce)
print("Scenario probabilities:", P)

All counties: 7
All locations: 43
Warehouse/Supplier locations: 6 [0, 1, 2, 3, 4, 5]
Shelter locations: 21 [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]
RSA locations: 9 [27, 28, 29, 30, 31, 32, 33, 34, 35]
PoD locations: 7 [36, 37, 38, 39, 40, 41, 42]
All resources: 6
Time periods: 18
Time Before Landfall: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Time After Landfall: [9, 10, 11, 12, 13, 14, 15, 16, 17]
Scenario indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
Scenario probabilities: [0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02]


In [23]:
#Create Arc Set where cannot go from shelter to PoD

A = set() #set to store arc pairs

no_demand=[]
for i in set(V1).union(set(V3)):
    i_has_demand = any(df_V['County'][i] == df_C['County'][c] and df_C['Shelter Total Demand'][c] > 0 for c in range(C))
    if not i_has_demand:
        no_demand.append(i)
print(no_demand)

for i in range(V):
    for j in range(V):
        if i != j and i not in no_demand and j not in no_demand:
            if (i not in V1 or j not in V3) and (i not in V3 or j not in V1):
                A.add((i, j))

#print("Arc set:", arc_set)
print("Number of arcs:", len(A))
print("Number of arcs eliminated by this arc set:", (V*V)-len(A))
#print(A)

[23, 42]
Number of arcs: 1400
Number of arcs eliminated by this arc set: 449


In [37]:
#Defining parameters

W_k = df_inputs['People on Truck'][0] #number of people on a truck
tp_salary = df_inputs['People on Truck'][0]*tperiod; # salary per time period 
fuel = df_inputs['Fuel Cost Per Mile'][0]*0.000621371; # fuel cost per meter
open_array = [df_inputs['Open Cost V0'][0],df_inputs['Open Cost V1'][0],df_inputs['Open Cost V2'][0],df_inputs['Open Cost V3'][0]] #Assumed opening costs

#Shelter Capacity at location i when i in V1 q_i[I]
q_i=[]
for i in V1:
    q_i.append(df_V['Pallet Capacity'][i])

#starting inventory I_ir0[I][R]
Inv=[]
for i in range(V):
    Inv.extend([df_V['S-Kit StartInv'][i],
                  df_V['P-Kit StartInv'][i],
                  df_V['Water StartInv'][i],
                  df_V['Food StartInv'][i],
                  df_V['Cot StartInv'][i],
                  df_V['Blanket StartInv'][i]])
    I_ir0 = [Inv[i:i+R] for i in range(0, len(Inv), R)]

#penalty cost for location i p_i[I]
p_i=[]
for i in range(V):
    p_i.append(df_V['P_i'][i])

#time periods to travel i,j t_ijt[i][j]
t_ijt=[]
for i in range(V):
    row=[]
    for j in range(V):
        t_ijt_hours=timeMatrix[i][j]/3600
        t_ijt_period=math.ceil(t_ijt_hours/tperiod)
        row.append(t_ijt_period)
    t_ijt.append(row)

#distance to travel i,j e_ij[i][j]
e_ij=[]
for i in range(V):
    row=[]
    for j in range(V):
        e_ij_miles=distanceMatrix[i][j]*0.000621371
        row.append(e_ij_miles)
    e_ij.append(row)

#truck delay h_i[i]
h_i=[]
for i in range(V):
    h_i.append(df_V['TruckDelay'][i])

#Fraction of a truck 1 pallet of resource r requires w_r[R]
w_r=[]
for r in range(R):
    w_r.append(1/df_R['Pallet Units on Truck'][r])

#Cost to procure resource r from location i cost_ir[i][r]
cost=[]
for i in V0:
    cost.extend([0,0,df_V['Water PC'][i],df_V['Food PC'][i],0,0])
    cost_ir = [cost[i:i+R] for i in range(0, len(cost), R)]
    
#Factor of pallet required per commodity r of kit g_shelter[R]
g_shelter=[]
for r in range(R):
    g_shelter.append(df_R['pallet per S-Kit'][r])

#Factor of pallet required per commodity r of kit g_pod[R]
g_pod=[]
for r in range(R):
    g_pod.append(df_R['pallet per P-Kit'][r])

# Initialize K_ijt as a nested dictionary {i: {j: {t: capacity}}}
K_ijt = {}

# Populate K_ijt with data from df_K
for index, row in df_K.iterrows():
    i = row['From_Node']
    j = row['To_Node']
    t = row['Time_Period']
    capacity = row['Capacity']
    
    if i not in K_ijt:
        K_ijt[i] = {}
    if j not in K_ijt[i]:
        K_ijt[i][j] = {}
    
    K_ijt[i][j][t] = capacity

# Check the first few entries in K_ijt to ensure it’s structured correctly
#print(K_ijt)
#print (Inv)
print(g_shelter)
print(g_pod)

[0.0, 0.0, 0.008928571428571428, 0.010416666666666666, 0.020833333333333332, 0.016666666666666666]
[0.0, 0.0, 0.008928571428571428, 0.010416666666666666, 0.0, 0.0]


In [27]:
# Define the model
m = gp.Model("Two-Stage Stochastic Programming Model")

# # Set the optimization direction
# m.modelSense = GRB.MINIMIZE

Set parameter Username
Academic license - for non-commercial use only - expires 2025-07-29


In [29]:
# Define decision variables 
# i, j is position, r is commodity, t is time, and s is scenario

# Define first-stage decision variables
# I: Inventory of commodity r at node i at the start of time period t
I = {}

for i in range(V):
    for r in range(R):
        for t in range(T):
            I[i, r, t] = m.addVar(vtype=GRB.CONTINUOUS, name="I_%s_%s_%s" % (i, r, t))

# Define second-stage decision variables
U = {}  # Demand shortage at location i at time period t in scenario s

for i in range(V):
    for t in range(T):
        for s in sce:  # Iterate directly over the scenario set (e.g., 0 and 1)
            U[i, t, s] = m.addVar(vtype=GRB.CONTINUOUS, name="U_%s_%s_%s" % (i, t, s))

S = {}  # Demand satisfied at location i at time period t in scenario s

for i in range(V):
    for t in range(T):
        for s in sce:  # Iterate directly over the scenario set
            S[i, t, s] = m.addVar(vtype=GRB.CONTINUOUS, name="S_%s_%s_%s" % (i, t, s))

f = {}  # Quantity of commodity r traveling on link (i, j) at time period t in scenario s

for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only define f_ijrt^s if (i, j) is a valid link
            for r in range(R):
                for t in range(T):
                    for s in sce:  # Iterate directly over the scenario set
                        f[i, j, r, t, s] = m.addVar(vtype=GRB.CONTINUOUS, name="f_%s_%s_%s_%s_%s" % (i, j, r, t, s))
m.update()

In [31]:
print(d_its.keys())


dict_keys([(0, 0, 0), (0, 1, 0), (0, 2, 0), (0, 3, 0), (0, 4, 0), (0, 5, 0), (0, 6, 0), (0, 7, 0), (0, 8, 0), (0, 9, 0), (0, 10, 0), (0, 11, 0), (0, 12, 0), (0, 13, 0), (0, 14, 0), (0, 15, 0), (0, 16, 0), (0, 17, 0), (1, 0, 0), (1, 1, 0), (1, 2, 0), (1, 3, 0), (1, 4, 0), (1, 5, 0), (1, 6, 0), (1, 7, 0), (1, 8, 0), (1, 9, 0), (1, 10, 0), (1, 11, 0), (1, 12, 0), (1, 13, 0), (1, 14, 0), (1, 15, 0), (1, 16, 0), (1, 17, 0), (2, 0, 0), (2, 1, 0), (2, 2, 0), (2, 3, 0), (2, 4, 0), (2, 5, 0), (2, 6, 0), (2, 7, 0), (2, 8, 0), (2, 9, 0), (2, 10, 0), (2, 11, 0), (2, 12, 0), (2, 13, 0), (2, 14, 0), (2, 15, 0), (2, 16, 0), (2, 17, 0), (3, 0, 0), (3, 1, 0), (3, 2, 0), (3, 3, 0), (3, 4, 0), (3, 5, 0), (3, 6, 0), (3, 7, 0), (3, 8, 0), (3, 9, 0), (3, 10, 0), (3, 11, 0), (3, 12, 0), (3, 13, 0), (3, 14, 0), (3, 15, 0), (3, 16, 0), (3, 17, 0), (4, 0, 0), (4, 1, 0), (4, 2, 0), (4, 3, 0), (4, 4, 0), (4, 5, 0), (4, 6, 0), (4, 7, 0), (4, 8, 0), (4, 9, 0), (4, 10, 0), (4, 11, 0), (4, 12, 0), (4, 13, 0), (4, 14,

In [41]:
# Define constraints

#Starting Value of I[t=0]
for i in range(V):
    for r in range(R):
        m.addConstr(I[i,r,0]==I_ir0[i][r])

# Update demand shortage for all locations, time periods (except t=0), and scenarios
for i in range(V):
    for t in range(1, T):
        for s in sce:
            m.addConstr(U[i, t, s] == U[i, t-1, s] - S[i, t, s] + d_its[(i, t, s)], 
                        name="DemandShortageUpdate_%s_%s_%s" % (i, t, s))

# Initial demand shortage for all locations and scenarios at time period t=0
for i in range(V):
    for s in sce:
        m.addConstr(U[i, 0, s] == d_its[(i, 0, s)], name="InitialDemandShortage_%s_%s" % (i, s))


# Demand satisfaction constraint for all locations, time periods, and scenarios
for i in range(V):
    for t in range(T):
        for s in sce:
            m.addConstr(S[i, t, s] <= U[i, t, s], name="DemandSatisfaction_%s_%s_%s" % (i, t, s))

# Inventory update constraint for all locations, commodities, time periods (except last), and scenarios
for i in range(V):
    for r in range(R):
        for t in range(T - 1):
            for s in sce:
                outgoing_flow = gp.quicksum(f[i, j, r, t, s] for j in range(V) if (i, j) in A)
                incoming_flow = gp.quicksum(f[j, i, r, t - t_ijt[j][i] + 1, s] for j in range(V) if (j, i) in A and t - t_ijt[j][i] + 1 >= 0)
                
                # Inventory update constraint for shelters
                if i in V1:
                    m.addConstr(
                        I[i, r, t + 1] == I[i, r, t] - outgoing_flow + incoming_flow - S[i, t, s] * g_shelter[r],
                        name="InventoryUpdate_Shelter_%s_%s_%s_%s" % (i, r, t, s)
                    )
                # Inventory update constraint for pods
                elif i in V3:
                    m.addConstr(
                        I[i, r, t + 1] == I[i, r, t] - outgoing_flow + incoming_flow - S[i, t, s] * g_pod[r],
                        name="InventoryUpdate_Pod_%s_%s_%s_%s" % (i, r, t, s)
                    )
                # Inventory update constraint for warehouses and RSAs
                else:
                    m.addConstr(
                        I[i, r, t + 1] == I[i, r, t] - outgoing_flow + incoming_flow,
                        name="InventoryUpdate_%s_%s_%s_%s" % (i, r, t, s)
                    )

# Inventory sufficiency constraint
for i in range(V):
    for r in range(R):
        for t in range(T):
            for s in sce:
                outgoing_flow = gp.quicksum(f[i, j, r, t, s] for j in range(V) if (i, j) in A)
                if i in V1:  # Shelter
                    m.addConstr(
                        I[i, r, t] >= outgoing_flow + S[i, t, s] * g_shelter[r],
                        name="InventorySufficiency_Shelter_%s_%s_%s_%s" % (i, r, t, s)
                    )
                elif i in V3:  # Pod
                    m.addConstr(
                        I[i, r, t] >= outgoing_flow + S[i, t, s] * g_pod[r],
                        name="InventorySufficiency_Pod_%s_%s_%s_%s" % (i, r, t, s)
                    )
                else:  # Warehouse or RSA
                    m.addConstr(
                        I[i, r, t] >= outgoing_flow,
                        name="InventorySufficiency_%s_%s_%s_%s" % (i, r, t, s)
                    )


# Flow capacity constraint for all links, time periods, and scenarios
for i in range(V):
    for j in range(V):
        if (i, j) in A:  # Only consider valid links in A
            for t in range(T):
                for s in sce:
                    # Check if the capacity for this (i, j, t) exists in K_ijt
                    if i in K_ijt and j in K_ijt[i] and t in K_ijt[i][j]:
                        m.addConstr(
                            gp.quicksum(w_r[r] * f[i, j, r, t, s] for r in range(R)) <= K_ijt[i][j][t],
                            name="FlowCapacity_%s_%s_%s_%s" % (i, j, t, s)
                        )
m.update()

In [43]:
U

{(0, 0, 0): <gurobi.Var U_0_0_0>,
 (0, 0, 1): <gurobi.Var U_0_0_1>,
 (0, 1, 0): <gurobi.Var U_0_1_0>,
 (0, 1, 1): <gurobi.Var U_0_1_1>,
 (0, 2, 0): <gurobi.Var U_0_2_0>,
 (0, 2, 1): <gurobi.Var U_0_2_1>,
 (0, 3, 0): <gurobi.Var U_0_3_0>,
 (0, 3, 1): <gurobi.Var U_0_3_1>,
 (0, 4, 0): <gurobi.Var U_0_4_0>,
 (0, 4, 1): <gurobi.Var U_0_4_1>,
 (0, 5, 0): <gurobi.Var U_0_5_0>,
 (0, 5, 1): <gurobi.Var U_0_5_1>,
 (0, 6, 0): <gurobi.Var U_0_6_0>,
 (0, 6, 1): <gurobi.Var U_0_6_1>,
 (0, 7, 0): <gurobi.Var U_0_7_0>,
 (0, 7, 1): <gurobi.Var U_0_7_1>,
 (0, 8, 0): <gurobi.Var U_0_8_0>,
 (0, 8, 1): <gurobi.Var U_0_8_1>,
 (0, 9, 0): <gurobi.Var U_0_9_0>,
 (0, 9, 1): <gurobi.Var U_0_9_1>,
 (0, 10, 0): <gurobi.Var U_0_10_0>,
 (0, 10, 1): <gurobi.Var U_0_10_1>,
 (0, 11, 0): <gurobi.Var U_0_11_0>,
 (0, 11, 1): <gurobi.Var U_0_11_1>,
 (0, 12, 0): <gurobi.Var U_0_12_0>,
 (0, 12, 1): <gurobi.Var U_0_12_1>,
 (0, 13, 0): <gurobi.Var U_0_13_0>,
 (0, 13, 1): <gurobi.Var U_0_13_1>,
 (0, 14, 0): <gurobi.Var U_0_14_

In [45]:
P_s

[0.5, 0.5]

In [55]:
# Set up the objective function
objective = gp.quicksum(
    P[s] * gp.quicksum(
        gp.quicksum(
            p_i[i] * U[i, t, s]
            for i in range(V)
        ) for t in range(T)
    )
    for s in sce  # Iterate over the defined scenario indices
)
m.update()

m.setObjective(objective, GRB.MINIMIZE)
m.update()


In [57]:
# Optimize the model
m.optimize()

if m.status == GRB.OPTIMAL:
    print(f"Optimal Objective Value: {m.ObjVal}")
    
    # Print the values of the decision variables
    #print("\nDecision Variable Values:")
    #for v in m.getVars():
     #   print(f"{v.VarName}: {v.X}")
else:
    print("The model did not solve to optimality. Status code:", m.status)


Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13800H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 30030 rows, 310140 columns and 942218 nonzeros
Model fingerprint: 0xa278d3d6
Coefficient statistics:
  Matrix range     [9e-03, 1e+00]
  Objective range  [6e-02, 4e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [8e+00, 1e+04]
Presolve removed 12047 rows and 47007 columns
Presolve time: 0.70s
Presolved: 17983 rows, 263133 columns, 819570 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 5.39s

Barrier statistics:
 AA' NZ     : 5.300e+05
 Factor NZ  : 1.014e+07 (roughly 200 MB of memory)
 Factor Ops : 9.816e+09 (less than 1 second per iteration)
 Threads    : 12

                  Objective                Residual
Iter       Primal          Dual 

In [59]:
# Solve the model
m.optimize()

# Check if the optimization was successful
if m.status == GRB.OPTIMAL:
    # Create an empty list to store variable values
    variable_data = []

    # Loop through decision variables and get their values
    for var in m.getVars():
        variable_data.append([var.VarName, var.X])  # Append variable name and value

    # Create a DataFrame from the collected data
    df = pd.DataFrame(variable_data, columns=['Variable', 'Value'])

    # Save the DataFrame to an Excel file
    output_file = 'decision_variables_output.xlsx'
    df.to_excel(output_file, index=False, engine='openpyxl')

    print(f'Decision variable values have been saved to {output_file}')
else:
    print("The model did not find an optimal solution.")


Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13800H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 30030 rows, 310140 columns and 942218 nonzeros
Coefficient statistics:
  Matrix range     [9e-03, 1e+00]
  Objective range  [6e-02, 4e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [8e+00, 1e+04]

Solved in 0 iterations and 0.02 seconds (0.00 work units)
Optimal objective  1.329138896e+05
Decision variable values have been saved to decision_variables_output.xlsx
